# 01 – EC Antitrust Master List

## Ziel dieses Notebooks

Dieses Notebook liest eine lokale JSON-Datei mit EC-Wettbewerbsfällen ein, filtert daraus ausschliesslich **Antitrust-Fälle** heraus und exportiert eine saubere, flache Mastertabelle.

### Warum ist die EC-Masterliste der erste Schritt?

Die Europäische Kommission (EC) ist die zentrale Behörde für Wettbewerbsrecht in der EU. Ihre Falldatenbank ist der Ausgangspunkt für alle weiteren Analysen:
- Ohne eine saubere EC-Liste können wir keine CJEU-Urteile zuordnen.
- Ohne klare Fall-IDs können wir keine Verknüpfungen herstellen.
- Ohne Filterung auf Antitrust würden Merger-, Beihilfe- und DMA-Fälle die Analyse verfälschen.



## 1 – Imports und Konfiguration


In [1]:
import json
import re
from pathlib import Path

import pandas as pd

# Pandas-Ausgabe etwas breiter stellen, damit Spalten lesbar bleiben
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)

print("Imports OK")


Imports OK


## 2 – Input- und Output-Pfade definieren

Hier legen wir fest, wo die Quelldatei liegt und wohin die Ergebnisse gespeichert werden.
Alle Pfade werden mit `pathlib.Path` definiert – das funktioniert auf Windows, Mac und Linux gleich.


In [2]:
# Pfad zur Quelldatei (JSON oder Excel)
# Source von: https://compcases-open-data-portal-files-prod.s3.eu-west-1.amazonaws.com/case-data-AT.json
INPUT_FILE = Path("data/case-data-AT.json")

# Ausgabeordner – wird automatisch erstellt, falls er nicht existiert
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ausgabedateien
OUTPUT_CSV     = OUTPUT_DIR / "ec_antitrust_master.csv"
OUTPUT_PARQUET = OUTPUT_DIR / "ec_antitrust_master.parquet"

print(f"Input : {INPUT_FILE.resolve()}")
print(f"Output: {OUTPUT_DIR.resolve()}")
print(f"Datei existiert: {INPUT_FILE.exists()}")


Input : C:\vsCodeWorkspace\masterZHAW\masterarbeit\data_analysis_master\data\case-data-AT.json
Output: C:\vsCodeWorkspace\masterZHAW\masterarbeit\data_analysis_master\data\processed
Datei existiert: True


## 3 – Daten laden

Die Quelldatei ist eine JSON-Datei, in der jeder Schlüssel eine EC-Fall-ID ist (z. B. `AT.39294`).
Der Wert ist ein Objekt mit `metadata`, `caseAttachments` und `decisions`.


In [3]:
def load_input_file(path: Path):
    """Lädt JSON – gibt ein dict (JSON) zurück."""
    suffix = path.suffix.lower()
    if suffix == ".json":
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        print(f"JSON geladen: {len(data)} Einträge (Top-Level-Keys)")
        return "json", data
    else:
        raise ValueError(f"Unbekanntes Dateiformat: {suffix}. Erwartet: .json")


file_type, raw_data = load_input_file(INPUT_FILE)


JSON geladen: 748 Einträge (Top-Level-Keys)


## 4 – Hilfsfunktionen

Die JSON-Felder sind fast immer **Listen** (auch wenn nur ein Wert drin ist).
Ausserdem sind manche Felder wie `caseSectors` oder `caseLegalBasis` als **JSON-Strings innerhalb der Liste** kodiert.

Diese kleinen Hilfsfunktionen machen den Code übersichtlich und robust.


In [5]:
def first_value(x):
    """Gibt den ersten Eintrag einer Liste zurück, oder None wenn leer/kein Wert."""
    if isinstance(x, list) and len(x) > 0:
        return x[0]
    return None


def join_values(x, sep="; "):
    """Verbindet alle Einträge einer Liste zu einem lesbaren String."""
    if isinstance(x, list):
        return sep.join(str(v) for v in x if v)
    return str(x) if x else ""


def safe_json_parse(x):
    """
    Versucht, einen JSON-String zu dekodieren.
    Wird für Felder wie caseSectors oder caseLegalBasis verwendet,
    die als JSON-String innerhalb der Liste gespeichert sind.
    Gibt bei Fehler den Originalwert zurück.
    """
    if isinstance(x, str):
        try:
            return json.loads(x)
        except (json.JSONDecodeError, ValueError):
            return x
    return x


def extract_labels(field_list):
    """
    Extrahiert 'label'-Werte aus einer Liste von JSON-kodierten Objekten.
    Beispiel: ['{"code":"X","label":"Art. 101 TFEU"}'] → 'Art. 101 TFEU'
    """
    labels = []
    if not isinstance(field_list, list):
        return ""
    for item in field_list:
        parsed = safe_json_parse(item)
        if isinstance(parsed, dict) and "label" in parsed:
            labels.append(parsed["label"])
        elif isinstance(parsed, str):
            labels.append(parsed)
    return "; ".join(labels)


def normalize_case_id(case_id):
    """
    Normalisiert eine EC-Fall-ID:
    - Whitespace entfernen
    - Grossbuchstaben
    - Konsistentes Format: AT.XXXXX
    Beispiel: ' at.39294 ' → 'AT.39294'

    Hinweis: Legacy-IDs (z. B. IV/...) können hier noch nicht vollständig
    normalisiert werden – das ist ein späterer Schritt.
    """
    if not case_id or not isinstance(case_id, str):
        return None
    normalized = case_id.strip().upper()
    # Sicherstellen, dass AT. korrekt geschrieben ist (kein Leerzeichen nach AT)
    normalized = re.sub(r'^AT\s*\.\s*', 'AT.', normalized)
    return normalized


print("Hilfsfunktionen definiert.")

# Kurze Tests
assert first_value(["a", "b"]) == "a"
assert first_value([]) is None
assert join_values(["x", "y"]) == "x; y"
assert normalize_case_id(" at.39294 ") == "AT.39294"
assert normalize_case_id("AT . 39294") == "AT.39294"
print("Alle Tests bestanden.")


Hilfsfunktionen definiert.
Alle Tests bestanden.


## 5 – Antitrust-Filterlogik

### Warum filtern wir?

Die EC-Datenbank enthält verschiedene Falltypen:
- **Antitrust** (Art. 101/102 TFEU) – das wollen wir
- **Cartels** – oft Teil von Antitrust, aber separat klassifiziert
- **Mergers** – Fusionskontrolle, nicht relevant
- **State Aid** – Beihilferecht, nicht relevant
- **DMA / FSR** – neuere Instrumente, nicht relevant

### Filterkriterien

Wir behalten einen Fall, wenn **mindestens eines** dieser Kriterien zutrifft:
1. `caseCartel` enthält `"Antitrust"` oder `"Cartel"` → primäres Kriterium
2. `caseInstrument` enthält `"Antitrust"` → sekundäres Kriterium

Fälle mit `caseInstrument` wie `"Merger"`, `"State Aid"`, `"DMA"`, `"FSR"` werden **ausgeschlossen**.


In [6]:
# Schlüsselwörter, die auf Antitrust hinweisen
ANTITRUST_KEYWORDS = {"antitrust", "cartel"}

# Schlüsselwörter, die explizit NICHT Antitrust sind
EXCLUDE_KEYWORDS = {"merger", "state aid", "dma", "fsr"}


def is_antitrust_case(metadata: dict) -> bool:
    """
    Entscheidet, ob ein Fall ein Antitrust-Fall ist.

    Gibt True zurück, wenn:
    - caseCartel oder caseInstrument einen Antitrust-Begriff enthält
    UND
    - kein Ausschluss-Begriff (Merger, State Aid, etc.) vorkommt
    """
    cartel_val = join_values(metadata.get("caseCartel", [])).lower()
    instr_val  = join_values(metadata.get("caseInstrument", [])).lower()
    combined   = cartel_val + " " + instr_val

    has_antitrust = any(kw in combined for kw in ANTITRUST_KEYWORDS)
    has_exclusion = any(kw in combined for kw in EXCLUDE_KEYWORDS)

    return has_antitrust and not has_exclusion


print("Filterlogik definiert.")

# Kurze Tests
assert is_antitrust_case({"caseCartel": ["Antitrust"], "caseInstrument": ["Antitrust & Cartels"]}) == True
assert is_antitrust_case({"caseCartel": ["Cartel"],   "caseInstrument": ["Antitrust & Cartels"]}) == True
assert is_antitrust_case({"caseCartel": [],            "caseInstrument": ["Merger"]})              == False
assert is_antitrust_case({"caseCartel": ["Antitrust"], "caseInstrument": ["Merger"]})              == False
print("Alle Filter-Tests bestanden.")


Filterlogik definiert.
Alle Filter-Tests bestanden.


## 6 – Flattening: Aus JSON-Struktur eine flache Tabelle bauen

Jeder Fall in der JSON-Datei hat eine verschachtelte Struktur.
Wir "flatten" diese Struktur zu einer einzigen Zeile pro Fall.

Die Funktion `extract_case_row()` macht genau das: Sie nimmt einen Fall und gibt ein Dictionary zurück,
das direkt als Zeile in einem DataFrame verwendet werden kann.


In [7]:
def extract_case_row(case_id: str, case_obj: dict) -> dict:
    """
    Extrahiert alle relevanten Felder aus einem Fall-Objekt
    und gibt eine flache Zeile als Dictionary zurück.

    Parameter:
        case_id    : der Schlüssel aus dem JSON (z. B. 'AT.39294')
        case_obj   : das vollständige Fall-Objekt mit metadata, caseAttachments, decisions
        source_file: Name der Quelldatei (für Nachvollziehbarkeit)
    """
    meta        = case_obj.get("metadata", {})
    attachments = case_obj.get("caseAttachments", [])
    decisions   = case_obj.get("decisions", [])

    # Rohe Fall-ID aus den Metadaten (kann von case_id abweichen)
    raw_case_id = first_value(meta.get("caseNumber", [])) or case_id

    row = {
        # Identifikation
        "ec_case_id"             : raw_case_id,
        "ec_case_id_normalized"  : normalize_case_id(raw_case_id),

        # Basisdaten
        "case_title"             : first_value(meta.get("caseTitle", [])),
        "case_type"              : first_value(meta.get("caseType", [])),
        "case_instrument"        : first_value(meta.get("caseInstrument", [])),
        "case_cartel"            : first_value(meta.get("caseCartel", [])),

        # Daten
        "case_initiation_date"   : first_value(meta.get("caseInitiationDate", [])),
        "case_last_decision_date": first_value(meta.get("caseLastDecisionDate", [])),

        # Verwaltung
        "case_dg"                : first_value(meta.get("caseDg", [])),
        "language"               : first_value(meta.get("language", [])),

        # Beteiligte Unternehmen (als pipe-separierter String)
        "companies_raw"          : join_values(meta.get("caseCompanies", []), sep=" | "),

        # Rechtsgrundlagen (Labels extrahieren)
        "legal_basis_raw"        : extract_labels(meta.get("caseLegalBasis", [])),

        # Sektoren (Labels extrahieren)
        "sectors_raw"            : extract_labels(meta.get("caseSectors", [])),

        # Anhänge und Entscheidungen
        "attachments_count"      : len(attachments) if isinstance(attachments, list) else 0,
        "decisions_count"        : len(decisions)   if isinstance(decisions, list)   else 0,
    }
    return row


print("extract_case_row() definiert.")


extract_case_row() definiert.


## 7 – Hauptverarbeitung: Alle Fälle durchlaufen, filtern und flattenen

Jetzt kombinieren wir alles:
1. Jeden Fall aus der JSON-Datei lesen
2. Prüfen, ob es ein Antitrust-Fall ist
3. Falls ja: in eine flache Zeile umwandeln
4. Alle Zeilen zu einem DataFrame zusammenführen


In [8]:
source_file_name = INPUT_FILE.name
rows = []
skipped_non_antitrust = 0
skipped_errors = 0

if file_type == "json":
    for case_id, case_obj in raw_data.items():
        try:
            meta = case_obj.get("metadata", {})

            # Antitrust-Filter anwenden
            if not is_antitrust_case(meta):
                skipped_non_antitrust += 1
                continue

            row = extract_case_row(case_id, case_obj)
            rows.append(row)

        except Exception as e:
            print(f"  WARNUNG: Fehler bei Fall '{case_id}': {e}")
            skipped_errors += 1


# DataFrame erstellen
df = pd.DataFrame(rows)

print(f"\nVerarbeitung abgeschlossen:")
print(f"  Antitrust-Fälle gefunden      : {len(df)}")
print(f"  Nicht-Antitrust übersprungen  : {skipped_non_antitrust}")
print(f"  Fehler beim Verarbeiten       : {skipped_errors}")



Verarbeitung abgeschlossen:
  Antitrust-Fälle gefunden      : 748
  Nicht-Antitrust übersprungen  : 0
  Fehler beim Verarbeiten       : 0


## 8 – Normalisierung der Datumsspalten

Wir konvertieren die Datumsspalten in ein einheitliches Format (`datetime`).
Fehlerhafte oder fehlende Werte werden dabei als `NaT` (Not a Time) gespeichert – das bricht das Notebook nicht ab.


In [9]:
for date_col in ["case_initiation_date", "case_last_decision_date"]:
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

# Leere Strings in None umwandeln (für saubere Null-Werte)
df = df.replace("", None)

print("Datumsspalten normalisiert.")
print(df[["ec_case_id", "case_initiation_date", "case_last_decision_date"]].head(5))


Datumsspalten normalisiert.
  ec_case_id case_initiation_date case_last_decision_date
0   AT.35803           1995-10-20                     NaT
1   AT.34950           1993-12-17              2001-06-15
2   AT.39172           2005-03-04              2007-01-10
3   AT.39294           2006-02-22                     NaT
4   AT.39173           2005-03-04              2007-01-10


## 9 – Qualitätschecks

Bevor wir exportieren, prüfen wir die Qualität der Daten:
- Wie viele Fälle haben wir?
- Wie viele Case IDs fehlen?
- Gibt es Dubletten?
- Wie verteilen sich die Falltypen?


In [10]:
print("=" * 60)
print("QUALITÄTSCHECKS")
print("=" * 60)

total_input = len(raw_data)
print(f"\nAnzahl Input-Fälle (gesamt in Datei) : {total_input}")
print(f"Anzahl Antitrust-Fälle (nach Filter) : {len(df)}")
print(f"Anteil Antitrust                     : {len(df) / total_input * 100:.1f}%")

# Fehlende Case IDs
missing_ids = df["ec_case_id"].isna().sum()
print(f"\nFehlende Case IDs (ec_case_id)       : {missing_ids}")

# Fehlende normalisierte IDs
missing_norm = df["ec_case_id_normalized"].isna().sum()
print(f"Fehlende normalisierte IDs           : {missing_norm}")

# Dubletten
dupes = df["ec_case_id_normalized"].duplicated().sum()
print(f"Dubletten (ec_case_id_normalized)    : {dupes}")

# Fehlende Titel
missing_title = df["case_title"].isna().sum()
print(f"Fehlende Falltitel                   : {missing_title}")

# Fehlende Initiierungsdaten
missing_date = df["case_initiation_date"].isna().sum()
print(f"Fehlende Initiierungsdaten           : {missing_date}")


QUALITÄTSCHECKS

Anzahl Input-Fälle (gesamt in Datei) : 748
Anzahl Antitrust-Fälle (nach Filter) : 748
Anteil Antitrust                     : 100.0%

Fehlende Case IDs (ec_case_id)       : 0
Fehlende normalisierte IDs           : 0
Dubletten (ec_case_id_normalized)    : 0
Fehlende Falltitel                   : 0
Fehlende Initiierungsdaten           : 0


In [11]:
print("\nHäufigkeitsverteilung: case_type")
print("-" * 40)
print(df["case_type"].value_counts(dropna=False).to_string())



Häufigkeitsverteilung: case_type
----------------------------------------
case_type
AtStandardATCCase      704
AtStateMeasureCase      38
AtSectorInquiryCase      6


In [12]:
print("\nHäufigkeitsverteilung: case_instrument")
print("-" * 40)
print(df["case_instrument"].value_counts(dropna=False).to_string())



Häufigkeitsverteilung: case_instrument
----------------------------------------
case_instrument
Antitrust & Cartels    748


In [13]:
print("\nHäufigkeitsverteilung: case_cartel")
print("-" * 40)
print(df["case_cartel"].value_counts(dropna=False).to_string())



Häufigkeitsverteilung: case_cartel
----------------------------------------
case_cartel
Antitrust    566
Cartel       182


In [15]:
print("\nÜbersicht aller Spalten und Datentypen:")
print(df.dtypes)
print(f"\nShape: {df.shape[0]} Zeilen × {df.shape[1]} Spalten")



Übersicht aller Spalten und Datentypen:
ec_case_id                            str
ec_case_id_normalized                 str
case_title                            str
case_type                             str
case_instrument                       str
case_cartel                           str
case_initiation_date       datetime64[us]
case_last_decision_date    datetime64[us]
case_dg                               str
language                              str
companies_raw                         str
legal_basis_raw                       str
sectors_raw                           str
attachments_count                   int64
decisions_count                     int64
has_attachments                      bool
has_decisions                        bool
source_file                           str
dtype: object

Shape: 748 Zeilen × 18 Spalten


## 10 – Export

Wir speichern die Mastertabelle in zwei Formaten:
- **CSV**: universell lesbar, gut für Excel und andere Tools
- **Parquet**: effizientes Binärformat, ideal für grosse Datensätze und spätere Python-Analysen

Beide Dateien landen im Ordner `data/processed/`.


In [16]:
# CSV exportieren
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"CSV gespeichert: {OUTPUT_CSV.resolve()}")
print(f"  Dateigrösse: {OUTPUT_CSV.stat().st_size / 1024:.1f} KB")

# Parquet exportieren (optional – benötigt pyarrow oder fastparquet)
try:
    df.to_parquet(OUTPUT_PARQUET, index=False)
    print(f"\nParquet gespeichert: {OUTPUT_PARQUET.resolve()}")
    print(f"  Dateigrösse: {OUTPUT_PARQUET.stat().st_size / 1024:.1f} KB")
except ImportError:
    print("\nHinweis: Parquet-Export übersprungen (pyarrow nicht installiert).")
    print("  Installation: pip install pyarrow")


CSV gespeichert: C:\vsCodeWorkspace\masterZHAW\masterarbeit\data_analysis_master\data\processed\ec_antitrust_master.csv
  Dateigrösse: 252.0 KB

Hinweis: Parquet-Export übersprungen (pyarrow nicht installiert).
  Installation: pip install pyarrow


## 11 – Nächste Schritte

Dieses Notebook hat den ersten Grundbaustein gelegt. Hier ist, was als nächstes kommt:

- **Dieses Notebook** erstellt die EC-Antitrust-Masterliste aus der lokalen JSON-Datei.
- **Nächster Schritt (02):** Ergänzung älterer EC-Fälle vor 1995, die möglicherweise in anderen Quellen oder mit Legacy-IDs (z. B. `IV/...`) vorliegen.
- **Danach (03):** Sammlung von CJEU-Fällen (Court of Justice der EU) aus dem EUR-Lex-System.
- **Danach (04):** Verknüpfung – prüfen, welche CJEU-Urteile EC-Antitrust-Fälle zitieren oder darauf basieren.
- **Später:** Volltext-Analyse der Entscheidungen (PDFs herunterladen und auswerten).
